<a href="https://colab.research.google.com/github/cantika-alff/2025_PBO_TI1B/blob/main/TUGAS_BESAR_PBO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS BESAR PBO : TravelNote : Biaya & Lokasi (Aplikasi Pencatatan Pengeluaran Perjalanan Terintegrasi dengan Informasi Lokasi Wisata)

## Dasar Teori : Konsep - Konsep Utama yang Digunakan dalam Pengembangan Aplikasi Ini.

In [ ]:
# 1. Pandas untuk Pengolahan Data CSV : Membaca Data dari file lokasi_wonogiri.csv
import pandas as pd
df_lokasi = pd.read_csv(‘data/lokasi_wonogiri.csv’)
print(df_lokasi.head())

# 2. Folium untuk Visualisasi Peta : Menampilkan data lokasi wisata dalam bentuk peta interaktif
import folium
m = folium.Map(location=[-7.9, 110.95], zoom_start=11)

folium.Marker(
    location=[-7.84699, 110.92518],
    popup="<b>Waduk Gajah Mungkur</b>",
    icon=folium.Icon(color="green", icon="tree")
).add_to(m)

m.save("peta_travelnote.html")

# 3. SQLite untuk Penyimpanan Data : Menyimpan data transaksi secara permanen dana mendukung operasi CRUD
CREATE TABLE transaksi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanggal TEXT,
    lokasi TEXT,
    kategori TEXT,
    nominal INTEGER,
    deskripsi TEXT
);

# 4. Framework Streamlit untuk Antarmuka : Antarmuka penggunan berbasis web dengan tampilan sederhana dan real-time
import streamlit as st
st.title("📍 TravelNote: Biaya & Lokasi")
st.date_input("Tanggal")
st.selectbox("Lokasi", list_lokasi)
st.number_input("Nominal", min_value=0)
st.text_area("Deskripsi")
st.button("Simpan")


## Langkah - Langkah Praktikum

### 1. Instalasi Library

In [ ]:
# Buat Virtual Environment
python  -m venv env

# Aktifkan Virtual Environment
env\Scripts\activate

# Install Library
pip install streamlit pandas folium

### 2. Setup DataBase SQLite

In [ ]:
# setup_db.py
import sqlite3
import os

os.makedirs("database", exist_ok=True)  # Jika ingin disimpan di folder
conn = sqlite3.connect("database/travelnote.db")  # atau "travelnote.db" kalau mau tetap di root
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS transaksi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanggal TEXT,
    lokasi TEXT,
    kategori TEXT,
    nominal INTEGER,
    keterangan TEXT
)
""")

conn.commit()
conn.close()
print("✅ Database setup selesai!")

# Jalankan File
python setup_db.py

### 3. Buat Model Data Transaksi

In [ ]:
from datetime import date

class Transaksi:
    def __init__(self, deskripsi, jumlah, kategori, lokasi, tanggal=None):
        self.deskripsi = deskripsi
        self.jumlah = float(jumlah)
        self.kategori = kategori
        self.lokasi = lokasi
        self.tanggal = tanggal or date.today()

    def to_tuple(self):

        return (
            self.tanggal.strftime("%Y-%m-%d"),
            self.lokasi,
            self.kategori,
            int(self.jumlah),
            self.deskripsi
        )

### 4. Buat Model Lokasi Wisata

In [ ]:
class Lokasi:
    def __init__(self, nama, lat, lon, tipe, deskripsi):
        self.nama = nama
        self.lat = lat
        self.lon = lon
        self.tipe = tipe
        self.deskripsi = deskripsi

    def get_icon(self):
        if self.tipe == "Wisata Alam":
            return "tree", "green"

        elif self.tipe == "Wisata Edukasi":
            return "book", "pink"

        elif self.tipe == "Kuliner":
            return "cutlery", "purple"

        elif self.tipe == "Tempat Ibadah":
            if "katolik" in self.nama.lower() or "gereja" in self.nama.lower():
                return "church", "orange"
            elif "masjid" in self.nama.lower():
                return "mosque", "orange"
            else:
                return "place-of-worship", "orange"

        else:
            return "map-marker", "blue"


def buat_lokasi(row):
    return Lokasi(
        nama=row["Nama"],
        lat=row["Latitude"],
        lon=row["Longitude"],
        tipe=row["Tipe"],
        deskripsi=row["Deskripsi"]
    )

### 5. Implementasi Modul Database

In [ ]:
import sqlite3
from modules.model_transaksi import Transaksi

def get_connection():
    return sqlite3.connect("database/travelnote.db")

def create_table():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS transaksi (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                tanggal TEXT,
                lokasi TEXT,
                kategori TEXT,
                nominal INTEGER,
                deskripsi TEXT
            )
        """)

def insert_transaksi(transaksi: Transaksi):
    with get_connection() as conn:
        conn.execute("""
            INSERT INTO transaksi (tanggal, lokasi, kategori, nominal, keterangan)
            VALUES (?, ?, ?, ?, ?)
        """, transaksi.to_tuple())

def get_all_transaksi():
    with get_connection() as conn:
        rows = conn.execute("SELECT * FROM transaksi").fetchall()
    return rows

def delete_transaksi_by_id(id_transaksi):
    with get_connection() as conn:
        conn.execute("DELETE FROM transaksi WHERE id = ?", (id_transaksi,))

### 6. Form Tambah Transaksi & Statistik

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from modules import database
from modules.model_transaksi import Transaksi
from config import KATEGORI_PENGELUARAN

def tambah_transaksi():
    st.subheader("Tambah Transaksi")

    df_lokasi = pd.read_csv("data/lokasi_wonogiri.csv")
    pilihan_lokasi = df_lokasi["Nama"].tolist()

    with st.form("form_tambah"):
        tanggal = st.date_input("Tanggal")
        lokasi = st.selectbox("Lokasi", pilihan_lokasi)
        kategori = st.selectbox("Kategori", ["Transportasi", "Makanan", "Tiket", "Belanja", "Lainnya"])
        nominal = st.number_input("Nominal", min_value=0)
        deskripsi = st.text_area("Deskripsi")
        submitted = st.form_submit_button("Simpan")

        if submitted:
            if not lokasi:
                st.warning("Lokasi belum dipilih.")
                return
            transaksi = Transaksi(deskripsi, nominal, kategori, lokasi, tanggal)
            database.insert_transaksi(transaksi)
            st.success("Transaksi berhasil ditambahkan")

def tampilkan_riwayat():
    st.subheader("Riwayat Transaksi")

    # ✅ Tampilkan pesan sukses jika ada di session_state
    if st.session_state.get("hapus_berhasil"):
        st.success(f"Transaksi ID {st.session_state['hapus_berhasil']} berhasil dihapus")
        del st.session_state["hapus_berhasil"]  # bersihkan flag agar tidak muncul terus

    data = database.get_all_transaksi()
    df = pd.DataFrame(data, columns=["ID", "Tanggal", "Lokasi", "Kategori", "Nominal", "Deskripsi"])
    st.dataframe(df)

    with st.form("form_hapus"):
        id_hapus = st.number_input("ID yang ingin dihapus", min_value=1, step=1)
        submit_hapus = st.form_submit_button("Hapus")

        if submit_hapus:
            ids_ada = df["ID"].tolist()
            if id_hapus in ids_ada:
                database.delete_transaksi_by_id(id_hapus)
                # ✅ simpan flag di session_state
                st.session_state["hapus_berhasil"] = id_hapus
                st.rerun()
            else:
                st.warning("ID tidak ditemukan.")

def tampilkan_statistik():
    st.subheader("📊 Statistik Pengeluaran")

    # Ambil dan siapkan data
    data = database.get_all_transaksi()
    df = pd.DataFrame(data, columns=["ID", "Tanggal", "Lokasi", "Kategori", "Nominal", "Deskripsi"])
    df["Nominal"] = pd.to_numeric(df["Nominal"], errors='coerce')
    df = df.dropna(subset=["Nominal"])

    if df.empty:
        st.warning("Belum ada data transaksi pengeluaran.")
        return

    kategori_total = df.groupby("Kategori")["Nominal"].sum()

    kategori = st.selectbox("Pilih Kategori", ["Semua"] + KATEGORI_PENGELUARAN)

    if kategori != "Semua":
        kategori_total = kategori_total[kategori_total.index == kategori]

    if kategori_total.empty:
        st.info("Tidak ada data untuk kategori ini.")
        return

    # Buat pie chart
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)  # ukuran seimbang, tidak terlalu besar

    result = ax.pie(
        list(kategori_total.astype(float)),
        labels=list(kategori_total.index.astype(str)),
        autopct='%1.1f%%',
        colors=["#F94144", "#F3722C", "#F9C74F", "#90BE6D", "#577590"],
        textprops={'fontsize': 12},
        wedgeprops=dict(width=0.5)  # donut style agar tidak terlihat penuh
    )

    # Tangani output pie
    if len(result) == 3:
        wedges, texts, autotexts = result
        for text in autotexts:
            text.set_fontsize(14)
    else:
        wedges, texts = result

    for label in texts:
        label.set_fontsize(14)

    ax.axis('equal')  # pastikan pie tetap bulat

    # Layout tengah supaya tidak full width
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.pyplot(fig)


### 7. Menampilkan Peta Lokasi Wisata

In [ ]:
import streamlit as st
import folium
import pandas as pd
import random
from streamlit_folium import st_folium
from modules.model_lokasi import buat_lokasi

# Memastikan data terbaru selalu dimuat (bukan cache lama)
def load_lokasi():
    return pd.read_csv("data/lokasi_wonogiri.csv")

def get_offsets_if_needed(df):
    if "offsets" not in st.session_state:
        st.session_state.offsets = {}
        used_coords = set()

        for idx, row in df.iterrows():
            latlon = (row["Latitude"], row["Longitude"])
            if latlon not in used_coords:
                st.session_state.offsets[idx] = (0, 0)
                used_coords.add(latlon)
            else:
                lat_offset = random.uniform(-0.002, 0.002)
                lon_offset = random.uniform(-0.002, 0.002)
                st.session_state.offsets[idx] = (lat_offset, lon_offset)
    return st.session_state.offsets

def generate_map(df, offsets):
    map_center = [-7.9, 110.9]
    m = folium.Map(location=map_center, zoom_start=11, width="100%", height="600px")

    for idx, row in df.iterrows():
        lokasi = buat_lokasi(row)
        icon_name, color = lokasi.get_icon()
        lat_offset, lon_offset = offsets[idx]

        lat = lokasi.lat + lat_offset
        lon = lokasi.lon + lon_offset

        # ✅ Tooltip tanpa koordinat
        tooltip = f"{lokasi.nama}"

        popup = f"""
            <b>{lokasi.nama}</b><br>
            <i>{lokasi.deskripsi}</i><br>
            <small>Lat: {lokasi.lat:.5f}, Lon: {lokasi.lon:.5f}</small>
        """

        folium.Marker(
            location=[lat, lon],
            popup=popup,
            tooltip=tooltip,
            icon=folium.Icon(icon=icon_name, color=color, prefix='fa')
        ).add_to(m)

    return m

def tampilkan_peta():
    st.subheader("📍 Peta Lokasi Wisata Wonogiri")
    df = load_lokasi()

    if df.empty:
        st.warning("Data lokasi kosong atau gagal dimuat.")
        return

    offsets = get_offsets_if_needed(df)
    peta = generate_map(df, offsets)

    if peta is None:
        st.error("Gagal membuat peta. Periksa data atau struktur kode.")
    else:
        st_folium(peta, width=1200, height=800)

# Jalankan langsung jika file ini dipanggil
if __name__ == "__main__":
    tampilkan_peta()


### 8. Main App untuk Menjalankan Aplikasi

In [ ]:
import streamlit as st
from modules.manajer_pengeluaran import tambah_transaksi, tampilkan_riwayat, tampilkan_statistik
from peta import tampilkan_peta
from modules.database import create_table

st.set_page_config(page_title="TravelNote", layout="wide", initial_sidebar_state="expanded")
create_table()
st.title("TravelNote: Biaya & Lokasi")

menu = st.sidebar.radio("Menu", ["Peta Lokasi", "Tambah Transaksi", "Riwayat Transaksi", "Statistik Pengeluaran"])

if menu == "Peta Lokasi":
    tampilkan_peta()
elif menu == "Tambah Transaksi":
    tambah_transaksi()
elif menu == "Riwayat Transaksi":
    tampilkan_riwayat()
elif menu == "Statistik Pengeluaran":
    tampilkan_statistik()


### 9. Menjalankan Aplikasi

In [ ]:
streamlit run main_app.py